# 03 · Steering

**Question.** Notebook 02 showed the daily and weekly phase are *readable* from the encoder. Does the forecast actually *use* them?

**Method.** Find each clock's plane at a layer, rotate every patch token in it by an angle δ mid-forward-pass, and measure how the forecast changes. Controls: the same rotation in random planes, and a calibration of the measuring tool.

- **Daily:** δ = π/2 is 6 h. Read off the forecast's daily phase.
- **Weekly:** δ = π/2 is 42 h. A week doesn't fit in the 64 h forecast, so compare the change against the exact change a 42 h shift would cause (known, since the data is synthetic).

## 1 · Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA

from torus_clock.model import capture, load_model, run_with_edit
from torus_clock.synthetic import P_DAY, P_WEEK, make_series

N_SERIES, T, H, NOISE, SEED = 300, 512, 64, 0.5, 0
DELTA = np.pi / 2                                    # quarter turn: 6 h daily, 42 h weekly
RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)

In [ ]:
model = load_model()
data = make_series(N_SERIES, T + H, noise=NOISE, seed=SEED)
context = torch.tensor(data["x"][:, :T])

acts, out = capture(model, context)
base = out.quantile_preds                             # [N_SERIES, 9, H], unedited forecast

PATCH = model.chronos_config.input_patch_size
N_PATCH, D = T // PATCH, model.config.d_model
LAYERS = [f"layer {i}" for i in range(len(model.encoder.block) + 1)]   # editable layers 0..6

patch_end = PATCH * np.arange(N_PATCH) + PATCH - 1
daily = data["daily_phase"][:, patch_end].reshape(-1)    # token labels, flattened
weekly = data["weekly_phase"][:, patch_end].reshape(-1)

## 2 · Tools

- `phase_plane`: average tokens in phase bins, PCA of the bin centroids gives the plane $U$ and centre $m$; `sign` fixes the orientation so $+\delta$ means later.
- `rotation`: $h' = h + U(R_\delta - I)\,U^\top(h - m)$, which rotates the in-plane part and leaves the rest untouched.
- `forecast_phase`: phase and amplitude of the median forecast's 24 h Fourier component over its first 48 h.

In [ ]:
def tokens(layer):
    return acts[layer][:, :N_PATCH].cpu().numpy().reshape(-1, D)


def phase_plane(A, phase, n_bins=24):
    bins = (phase / (2 * np.pi) * n_bins).astype(int) % n_bins
    C = np.stack([A[bins == b].mean(0) for b in range(n_bins)])
    m = C.mean(0)
    U = PCA(2).fit(C - m).components_.T
    xy = (C - m) @ U
    ang = np.arctan2(xy[:, 1], xy[:, 0])
    ctr = (np.arange(n_bins) + 0.5) * 2 * np.pi / n_bins
    sign = 1 if abs(np.exp(1j * (ang - ctr)).mean()) > abs(np.exp(1j * (ang + ctr)).mean()) else -1
    return U, m, sign


def random_plane(seed):
    return np.linalg.qr(np.random.default_rng(seed).normal(size=(D, 2)))[0]


def rotation(U, m, sign, delta):
    c, s = np.cos(sign * delta), np.sin(sign * delta)
    M, U, m = (torch.tensor(a, dtype=torch.float32) for a in ([[c - 1, -s], [s, c - 1]], U, m))
    return lambda h: h + ((h - m) @ U @ M.T) @ U.T


def steered(layer, U, m, sign, delta=DELTA):
    return run_with_edit(model, context, layer, rotation(U, m, sign, delta)).quantile_preds


def forecast_phase(q):
    f = q[:, 4, :48].numpy()
    F = (f - f.mean(1, keepdims=True)) @ np.exp(-2j * np.pi * np.arange(48) / P_DAY)
    return np.angle(F), np.abs(F)


wrap = lambda a: np.angle(np.exp(1j * a))
hours = lambda a: a * P_DAY / (2 * np.pi)
circ_mean = lambda a: np.angle(np.exp(1j * a).mean())

phase0, amp0 = forecast_phase(base)

## 3 · Calibration

Before trusting the ruler: a window ending $k$ hours earlier must move the forecast's daily phase by exactly $-k$ hours. No editing involved.

In [ ]:
for k in (3, 6, 12):
    moved = wrap(forecast_phase(model(context[:, : T - k]).quantile_preds.detach())[0] - phase0)
    print(f"{k:>2} h earlier: moved {hours(circ_mean(moved)):6.2f} h, "
          f"fidelity {np.cos(moved + 2 * np.pi * k / P_DAY).mean():.3f}")

## 4 · Daily steering

**fidelity** = mean $\cos(\text{moved} - \delta)$: 1 means every forecast moved by exactly δ, 0 means unrelated or no movement.

In [ ]:
def daily_effect(q):
    p, a = forecast_phase(q)
    moved = wrap(p - phase0)
    return np.cos(moved - DELTA).mean(), hours(circ_mean(moved)), np.median(a / amp0)


rows = []
for layer in LAYERS:
    U, m, sign = phase_plane(tokens(layer), daily)
    fid, h, amp = daily_effect(steered(layer, U, m, sign))
    rfid, rh, _ = daily_effect(steered(layer, random_plane(0), m, 1))
    rows.append(dict(layer=layer, fidelity=fid, hours=h, amplitude=amp, random_fidelity=rfid, random_hours=rh))

daily_steer = pd.DataFrame(rows).set_index("layer")
daily_steer.to_csv(RESULTS / "03_daily_steering.csv")
daily_steer.round(3)

In [ ]:
# one series: forecast before and after moving the daily clock 6 h forward at layer 3
k, layer = 0, "layer 3"
q = steered(layer, *phase_plane(tokens(layer), daily))
t = np.arange(T, T + H)

plt.figure(figsize=(12, 3.5))
plt.plot(np.arange(T - 72, T + H), data["x"][k, T - 72:T + H], color="gray", lw=0.8, label="data")
plt.plot(t, base[k, 4], label="forecast")
plt.plot(t, q[k, 4], label=f"daily clock +6 h ({layer})")
plt.axvline(T, color="black", lw=0.5)
plt.xlabel("hour"); plt.legend()
plt.tight_layout(); plt.savefig(RESULTS / "03_daily_example.png", dpi=150); plt.show()

## 5 · Weekly steering

Compare, for every series and forecast hour, the **realised** change with the **intended** one, $a_i w_i\,[\cos(\omega_t + \delta) - \cos\omega_t]$.

- **slope**: least-squares factor, realised ≈ slope × intended. 1 = the full shift.
- **corr**: how purely the change is a day shift. 1 = nothing else changed.
- **daily drift**: how far the time of day moved as a side effect (h).

In [ ]:
omega = data["weekly_phase"][:, T:T + H]
scale = (data["amp"] * data["weekly_amp"])[:, None]
intended = scale * (np.cos(omega + DELTA) - np.cos(omega))     # [N_SERIES, H]


def weekly_effect(q):
    realised = (q[:, 4] - base[:, 4]).numpy()
    slope = (realised * intended).sum() / (intended ** 2).sum()
    corr = np.corrcoef(realised.ravel(), intended.ravel())[0, 1]
    drift = hours(np.median(np.abs(wrap(forecast_phase(q)[0] - phase0))))
    return slope, corr, drift


rows, steered_weekly = [], {}
for layer in LAYERS:
    U, m, sign = phase_plane(tokens(layer), weekly, n_bins=28)
    steered_weekly[layer] = q = steered(layer, U, m, sign)
    s, c, d = weekly_effect(q)
    rs, rc, _ = weekly_effect(steered(layer, random_plane(0), m, 1))
    rows.append(dict(layer=layer, slope=s, corr=c, daily_drift=d, random_slope=rs, random_corr=rc))

weekly_steer = pd.DataFrame(rows).set_index("layer")
weekly_steer.to_csv(RESULTS / "03_weekly_steering.csv")
weekly_steer.round(3)

In [ ]:
# realised vs intended change: a wide cloud where steering fails, a tight diagonal band where it works
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True, sharey=True)
lim = np.abs(intended).max()
for ax, layer in zip(axes, ["layer 0", "layer 1", "layer 5"]):
    realised = (steered_weekly[layer][:, 4] - base[:, 4]).numpy()
    ax.scatter(intended.ravel(), realised.ravel(), s=1, alpha=0.2)
    ax.plot([-lim, lim], [-lim, lim], color="black", lw=0.8)
    r = weekly_steer.loc[layer]
    ax.set_title(f"{layer}: slope {r['slope']:.2f}, corr {r['corr']:.2f}")
    ax.set_xlabel("intended change")
axes[0].set_ylabel("realised change")
plt.tight_layout(); plt.savefig(RESULTS / "03_weekly_scatter.png", dpi=150); plt.show()

In [ ]:
# one series: weekly clock +42 h at layer 5, against what a true 42 h day shift would give
k, layer = 0, "layer 5"
plt.figure(figsize=(12, 3.5))
plt.plot(t, base[k, 4], label="forecast")
plt.plot(t, steered_weekly[layer][k, 4], label=f"weekly clock +42 h ({layer})")
plt.plot(t, base[k, 4].numpy() + intended[k], "--", color="black", lw=0.8, label="forecast + true 42 h shift")
plt.xlabel("hour"); plt.legend()
plt.tight_layout(); plt.savefig(RESULTS / "03_weekly_example.png", dpi=150); plt.show()

## 6 · Summary figure

In [ ]:
x = np.arange(len(LAYERS))
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].plot(x, daily_steer["fidelity"], "-o", label="clock plane")
axes[0].plot(x, daily_steer["random_fidelity"], "--s", label="random plane")
axes[0].set_title("daily, +6 h: fidelity")
axes[1].plot(x, weekly_steer["corr"], "-o", label="corr")
axes[1].plot(x, weekly_steer["slope"], "-^", label="slope")
axes[1].plot(x, weekly_steer["random_corr"], "--s", label="random corr")
axes[1].set_title("weekly, +42 h")
for ax in axes:
    ax.set_xticks(x, [l.split()[-1] for l in LAYERS]); ax.set_xlabel("layer"); ax.legend()
plt.tight_layout(); plt.savefig(RESULTS / "03_steering_by_layer.png", dpi=150); plt.show()

## 7 · Takeaways

Run on 14 November (`NOISE = 0.5`, `SEED = 0`).

- **The ruler is calibrated.** Windows ending 3, 6, 12 h earlier move the forecast by −2.99, −5.98, −11.96 h.
- **Daily steering works at every layer** (fidelity ≈ 0.99, ≈ 5.9 h of 6), layer 0 included. Expected: the daily circle is already in the input patch, so this is close to editing the input. Random planes: nothing.
- **Weekly steering only works where the model built the clock.** Layer 0 fails (corr 0.31, and the time of day moves 5 h); layer 1 is partial; layers 3–5 give slope 0.86–0.93, corr 0.85–0.93, with the time of day moving by under 0.3 h. Random planes: slope ≈ 0.
- **So:** the encoder builds a weekly clock, keeps it separate from the daily one, and the forecast reads it. Turning it makes the model forecast a different day.
- **Caveats.** Synthetic data; one model; a random plane is an easy control (a variance-matched plane would be stricter).

**Next:** real electricity load.